<a href="https://colab.research.google.com/github/Ayseatci/DI725_Project/blob/dev/notebooks/04_phase3/SWINV2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Multimodal Land Cover Regression – SwinV2 Experiments

This notebook implements a multimodal regression framework that combines image features with textual descriptions to predict land cover distributions.

The experiments are designed to evaluate the contribution of textual information to visual models and the effect of different caption types listed in the following:
  - Text-only captions (text_qwen3-4b)
  - Hybrid captions (hybrid_gemma3-4b)
  - Vision-generated captions (vision_gemma3-4b)
The impact of different lightweight text encoders (MiniLM,DistilBERT, RemoteCLIP) are also observed.

The image backbone used in this notebook is SwinV2, which is fine-tuned during training. Text encoders are kept frozen, and features are combined using a gated fusion mechanism.

All experiments are run under a consistent setup (same split, seed, training configuration) to ensure fair comparison.

## Environment Setup

This section installs the required libraries for the experiments.
These packages provide the core functionality for building and training the multimodal model.

In [ ]:
!pip install -q timm transformers wandb open_clip_torch huggingface_hub

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 23.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 5.0 MB/s eta 0:00:00


## Data Loading and Setup

The dataset is loaded from Google Drive and extracted locally. This setup ensures that images and their corresponding textual descriptions can be accessed efficiently during training.

In [ ]:
import os
import re
import random
import numpy as np
import pandas as pd
from PIL import Image

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

import timm
from transformers import AutoTokenizer, AutoModel
from sklearn.metrics import mean_absolute_error, mean_squared_error
from sklearn.model_selection import train_test_split

from torchvision import transforms

import wandb
import open_clip
from huggingface_hub import hf_hub_download
from google.colab import drive

In [ ]:
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
os.makedirs("/content/drive/MyDrive/2444347_DI725_term_project_phase3_predictions", exist_ok=True)

In [ ]:
!unzip -q /content/drive/MyDrive/DI725_project_dataset_nomask.zip

In [ ]:
LOCAL_DATA_ROOT = "/content/DI725_project_dataset_nomask"
CSV_PATH = os.path.join(LOCAL_DATA_ROOT, "captions.csv")
IMAGE_DIR = os.path.join(LOCAL_DATA_ROOT, "images")

os.makedirs(LOCAL_DATA_ROOT, exist_ok=True)

print("CSV:", CSV_PATH)
print("Images:", IMAGE_DIR)
print("Number of images:", len(os.listdir(IMAGE_DIR)))

CSV: /content/DI725_project_dataset_nomask/captions.csv
Images: /content/DI725_project_dataset_nomask/images
Number of images: 10000


## Experiment Configuration

This cell defines the main experimental settings used throughout the notebook, including sample size, image size, batch size, number of epochs, learning rate, weight decay, early stopping patience, and random seed.

The target variables are the seven classes: Tree, Shrub, Grass, Crop, Built-up, Barren, and Water. The text_scale parameter controls the strength of the textual contribution during gated fusion.

In [ ]:
CONFIG = {
    "sample_size": 10000,
    "img_size": 224,

    "batch_size": 32,
    "epochs": 15,
    "lr": 1e-4,
    "weight_decay": 1e-4,

    "model_name": "swinv2_tiny_window8_256",

    "early_stopping_patience": 3,
    "grad_clip": 1.0,

    "device": "cuda" if torch.cuda.is_available() else "cpu",
    "seed": 42,

    "text_scale": 0.7,

    # DataLoader
    "num_workers": 2,

    # Dataset
    "max_token_len": 128,

    # Model
    "regressor_hidden_dim": 256,
    "regressor_dropout": 0.25,

    # W&B
    "wandb_project": "DI725-Project_phase3_experiments",
    "wandb_project_scale": "DI725-Project_phase3_text_scale_exp",

    # Predictions output path
    "predictions_dir": "/content/drive/MyDrive/2444347_DI725_term_project_phase3_predictions",
}

TARGET_COLS = ["Tree", "Shrub", "Grass", "Crop", "Built-up", "Barren", "Water"]

## Reproducibility Setup

This cell fixes the random seed across Python, NumPy, and PyTorch to improve reproducibility. Using the same seed helps ensure that sampling, data splitting, and model training are comparable across experiments.

In [ ]:
def seed_everything(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

seed_everything(CONFIG["seed"])

## Data Loading and Text Cleaning

This section loads the caption and target data from the CSV file and prepares the text columns used in the multimodal experiments.

A cleaning function removes percentages, standalone numbers, and leakage related phrases from captions. This step prevents the model from directly using numerical target information contained in the text, while preserving semantic descriptions.

In [ ]:
df = pd.read_csv(CSV_PATH)

TEXT_COLUMNS_RAW = [
    "hybrid_gemma3-4b",
    "text_qwen3-4b",
    "vision_gemma3-4b"
]

def clean_caption_no_numbers(text):
    text = str(text).lower()

    # remove percentages: 53%, 53 percent, 53 percentage
    text = re.sub(r"\b\d+(\.\d+)?\s*%|\b\d+(\.\d+)?\s*percent(age)?", "", text)

    # remove standalone numbers
    text = re.sub(r"\b\d+(\.\d+)?\b", "", text)

    leakage_words = [
        "covering", "coverage", "accounts for", "occupies",
        "making up", "comprises", "constitutes", "represents",
        "estimated", "approximately", "around", "about"
    ]

    for word in leakage_words:
        text = text.replace(word, "")

    text = re.sub(r"[%(),:;]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()

    return text


for col in TEXT_COLUMNS_RAW:
    clean_col = col.replace("-", "_") + "_clean"
    df[clean_col] = df[col].apply(clean_caption_no_numbers)

    print(clean_col)
    print(
        "Remaining numeric leakage:",
        df[clean_col].str.contains(
            r"\d+\s*%|\d+\s*percent|\b\d+(\.\d+)?\b",
            regex=True,
            case=False
        ).sum()
    )

hybrid_gemma3_4b_clean
Remaining numeric leakage: 0


/tmp/ipykernel_1896/2462767828.py:40: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  df[clean_col].str.contains(


text_qwen3_4b_clean
Remaining numeric leakage: 0


/tmp/ipykernel_1896/2462767828.py:40: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  df[clean_col].str.contains(


vision_gemma3_4b_clean
Remaining numeric leakage: 0


/tmp/ipykernel_1896/2462767828.py:40: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  df[clean_col].str.contains(


In [ ]:
TEXT_COLS_CLEAN = [
    "hybrid_gemma3_4b_clean",
    "text_qwen3_4b_clean",
    "vision_gemma3_4b_clean"
]

## Stratified Sampling and Data Split

This section prepares the dataset for training by first filtering out rows with missing values in required columns (images, targets, and cleaned captions).

A dominant class is computed for each sample based on the maximum target value across land-cover classes. This is used to perform stratified sampling, ensuring that the class distribution is preserved.

## Train / Validation / Test Split

The sampled dataset is split into 80% training, 10% validation, 10% test

Stratification is applied based on the dominant class to maintain consistent class distribution across splits. This ensures fair and reliable evaluation of model performance.

Indices are reset after splitting to avoid indexing issues during data loading.

In [ ]:
needed_cols = (
    ["filename"]
    + TARGET_COLS
    + ["hybrid_gemma3_4b_clean", "text_qwen3_4b_clean", "vision_gemma3_4b_clean"]
)

df = df.dropna(subset=needed_cols).copy()

df["dominant_class"] = df[TARGET_COLS].idxmax(axis=1)

# If available rows <= sample_size, use all rows
if len(df) <= CONFIG["sample_size"]:
    sample_df = df.sample(frac=1.0, random_state=CONFIG["seed"]).reset_index(drop=True)
else:
    sample_df, _ = train_test_split(
        df,
        train_size=CONFIG["sample_size"],
        stratify=df["dominant_class"],
        random_state=CONFIG["seed"]
    )
    sample_df = sample_df.reset_index(drop=True)

print("Sample size:", len(sample_df))

train_df, temp_df = train_test_split(
    sample_df,
    test_size=0.20,
    stratify=sample_df["dominant_class"],
    random_state=CONFIG["seed"]
)

val_df, test_df = train_test_split(
    temp_df,
    test_size=0.50,
    stratify=temp_df["dominant_class"],
    random_state=CONFIG["seed"]
)

train_df = train_df.reset_index(drop=True)
val_df = val_df.reset_index(drop=True)
test_df = test_df.reset_index(drop=True)

print("Train:", len(train_df))
print("Val:", len(val_df))
print("Test:", len(test_df))

Sample size: 10000
Train: 8000
Val: 1000
Test: 1000


## Target Distribution Check

To verify that stratification is effective, this section computes summary statistics (mean and standard deviation) for each target class across the train, validation, and test splits.

This check ensures that the distributions of land-cover classes remain consistent across splits, preventing bias in training or evaluation.

In [ ]:
def check_target_distribution(train_df, val_df, test_df, target_cols):
    summary = []

    for name, data in [
        ("train", train_df),
        ("val", val_df),
        ("test", test_df)
    ]:
        row = {"split": name, "n": len(data)}

        for col in target_cols:
            row[f"{col}_mean"] = data[col].mean()
            row[f"{col}_std"] = data[col].std()

        summary.append(row)

    return pd.DataFrame(summary)

dist_summary = check_target_distribution(train_df, val_df, test_df, TARGET_COLS)
dist_summary

,split,n,Tree_mean,Tree_std,Shrub_mean,Shrub_std,Grass_mean,Grass_std,Crop_mean,Crop_std,Built-up_mean,Built-up_std,Barren_mean,Barren_std,Water_mean,Water_std
0,train,8000,28.8445,35.170320,0.828875,4.064469,45.367375,33.938682,17.95475,30.544088,1.139625,5.456130,4.192875,11.363640,1.672,9.244490
1,val,1000,28.8610,35.001154,0.818000,3.981297,45.064000,34.437681,18.15700,30.993789,1.190000,4.985053,4.181000,11.631231,1.729,9.926837
2,test,1000,28.1980,34.791741,0.941000,4.581103,45.318000,34.315075,18.14000,30.799334,1.095000,4.951615,4.508000,12.302898,1.800,9.461793


In [ ]:
dominant_dist = pd.DataFrame({
    "train": train_df["dominant_class"].value_counts(normalize=True),
    "val": val_df["dominant_class"].value_counts(normalize=True),
    "test": test_df["dominant_class"].value_counts(normalize=True)
}).fillna(0)

dominant_dist

,train,val,test
dominant_class,,,
Grass,0.470375,0.470,0.470
Tree,0.303750,0.304,0.303
Crop,0.180250,0.181,0.180
Barren,0.019250,0.019,0.020
Water,0.017750,0.018,0.018
Built-up,0.005875,0.006,0.006
Shrub,0.002750,0.002,0.003


## Image Transformations

This section defines preprocessing pipelines for training and evaluation.

Training transformations include resizing, random horizontal and vertical flips for data augmentation, and normalization using ImageNet statistics.

Evaluation transformations apply resizing and normalization only, ensuring consistent input without augmentation during validation and testing.

In [ ]:
train_tfms = transforms.Compose([
    transforms.Resize((CONFIG["img_size"], CONFIG["img_size"])),
    transforms.RandomHorizontalFlip(),
    transforms.RandomVerticalFlip(),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], # mean and sdt values of ImageNet channel
                         std=[0.229, 0.224, 0.225])
])

eval_tfms = transforms.Compose([
    transforms.Resize((CONFIG["img_size"], CONFIG["img_size"])),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225])
])

## Multimodal Dataset

This dataset class prepares inputs for multimodal training. For each sample the image is loaded and transformed and target values are extracted as a multi-output regression vector.
If a text column is provided, captions are tokenized using a pretrained tokenizer

The dataset returns, image tensor, target vector, tokenized text inputs. This structure enables joint processing of image and text features in the model.

In [ ]:
class LandCoverDataset(Dataset):
    def __init__(self, df, image_dir, transform=None, tokenizer=None, text_col=None, max_len=CONFIG["max_token_len"]):
        self.df = df.reset_index(drop=True)
        self.image_dir = image_dir
        self.transform = transform
        self.tokenizer = tokenizer
        self.text_col = text_col
        self.max_len = max_len

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]

        img_path = os.path.join(self.image_dir, row["filename"])
        image = Image.open(img_path).convert("RGB")

        if self.transform:
            image = self.transform(image)

        target = torch.tensor(
            row[TARGET_COLS].values.astype(np.float32),
            dtype=torch.float32
        )

        item = {
            "image": image,
            "target": target
        }

        if self.tokenizer is not None and self.text_col is not None:
            text = str(row[self.text_col])

            enc = self.tokenizer(
                text,
                padding="max_length",
                truncation=True,
                max_length=self.max_len,
                return_tensors="pt"
            )

            item["input_ids"] = enc["input_ids"].squeeze(0)
            item["attention_mask"] = enc["attention_mask"].squeeze(0)

        return item

In [ ]:
class LandCoverRawTextDataset(Dataset):
    def __init__(self, df, image_dir, transform=None, text_col=None):
        self.df = df.reset_index(drop=True)
        self.image_dir = image_dir
        self.transform = transform
        self.text_col = text_col

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]

        img_path = os.path.join(self.image_dir, row["filename"])
        image = Image.open(img_path).convert("RGB")

        if self.transform:
            image = self.transform(image)

        text = str(row[self.text_col])

        target = torch.tensor(
            row[TARGET_COLS].values.astype(np.float32),
            dtype=torch.float32
        )

        return {
            "image": image,
            "text": text,
            "target": target
        }

## Image Only Baseline Model

This model serves as a baseline using only visual information.

A pretrained vision backbone (SwinV2) is used to extract image features, which are then passed through a regression head to predict land-cover targets.

The regression head consists of layer normalization, fully connected layers,ReLU activation and dropout for regularization

This baseline allows comparison against multimodal models to measure the contribution of textual information. The backbone is initialized with pretrained weights and fine-tuned during training.

In [ ]:
class ImageOnlyModel(nn.Module):
    def __init__(self, model_name, num_outputs=len(TARGET_COLS)):
        super().__init__()

        self.backbone = timm.create_model(
            model_name,
            pretrained=True,
            num_classes=0
        )

        image_dim = self.backbone.num_features

        hidden = CONFIG["regressor_hidden_dim"]
        self.regressor = nn.Sequential(
            nn.LayerNorm(image_dim),
            nn.Linear(image_dim, hidden),
            nn.ReLU(),
            nn.Dropout(CONFIG["regressor_dropout"]),
            nn.Linear(hidden, num_outputs)
        )

    def forward(self, image):
        image_features = self.backbone(image)
        return self.regressor(image_features)

## Multimodal Model (Image + Text Fusion)

This model extends the image-only baseline by incorporating textual information through a gated fusion mechanism.

The architecture consists of:
- A pretrained vision backbone (SwinV2) for image feature extraction
- A pretrained text encoder (MiniLM / DistilBERT) for caption representation
- A projection layer to map text features into the image feature space

The text encoder is kept frozen during training to reduce computational cost and ensure stable representations.

To combine modalities, a gating mechanism is used. Image and text features are concatenated and passed through a learnable gate. The gate controls how much textual information contributes to the final representation. Moreover, a scaling factor further adjusts the strength of text features

The final fused representation is computed as:

    fused = image_features + scale × gate × text_features

This design allows the model to adaptively use textual information depending on its relevance.

In [ ]:
class ImageTextGatedFrozenScaledModel(nn.Module):
    def __init__(
        self,
        image_model_name,
        text_model_name,
        num_outputs=len(TARGET_COLS),
        text_scale=CONFIG["text_scale"]
    ):
        super().__init__()

        self.text_scale = text_scale

        self.image_backbone = timm.create_model(
            image_model_name,
            pretrained=True,
            num_classes=0
        )

        image_dim = self.image_backbone.num_features

        self.text_backbone = AutoModel.from_pretrained(text_model_name)

        for p in self.text_backbone.parameters():
            p.requires_grad = False

        text_dim = self.text_backbone.config.hidden_size

        self.text_proj = nn.Linear(text_dim, image_dim)

        self.gate_layer = nn.Sequential(
            nn.Linear(image_dim + image_dim, image_dim),
            nn.Sigmoid()
        )

        hidden = CONFIG["regressor_hidden_dim"]
        self.regressor = nn.Sequential(
            nn.LayerNorm(image_dim),
            nn.Linear(image_dim, hidden),
            nn.ReLU(),
            nn.Dropout(CONFIG["regressor_dropout"]),
            nn.Linear(hidden, num_outputs)
        )

    def mean_pooling(self, model_output, attention_mask):
        token_embeddings = model_output.last_hidden_state
        mask = attention_mask.unsqueeze(-1).expand(token_embeddings.size()).float()

        return torch.sum(token_embeddings * mask, dim=1) / torch.clamp(
            mask.sum(dim=1),
            min=1e-9
        )

    def forward(self, image, input_ids, attention_mask):
        image_features = self.image_backbone(image)

        with torch.no_grad():
            text_output = self.text_backbone(
                input_ids=input_ids,
                attention_mask=attention_mask
            )

        text_features = self.mean_pooling(text_output, attention_mask)
        text_features = self.text_proj(text_features)

        gate_input = torch.cat([image_features, text_features], dim=1)
        gate = self.gate_layer(gate_input)

        fused_features = image_features + self.text_scale * gate * text_features

        return self.regressor(fused_features)

In [ ]:
class RemoteCLIPTextEncoder(nn.Module):
    def __init__(self, model_name="ViT-B-32"):
        super().__init__()

        self.model, _, _ = open_clip.create_model_and_transforms(
            model_name,
            pretrained=None
        )

        self.tokenizer = open_clip.get_tokenizer(model_name)

        ckpt_path = hf_hub_download(
            repo_id="chendelong/RemoteCLIP",
            filename=f"RemoteCLIP-{model_name}.pt"
        )

        ckpt = torch.load(ckpt_path, map_location="cpu")

        if "state_dict" in ckpt:
            ckpt = ckpt["state_dict"]

        cleaned = {}
        for k, v in ckpt.items():
            k = k.replace("module.", "").replace("model.", "")
            cleaned[k] = v

        self.model.load_state_dict(cleaned, strict=False)

        print("RemoteCLIP weights loaded")

        for p in self.model.parameters():
            p.requires_grad = False

        self.model.eval()

    def forward(self, texts):
        device = next(self.model.parameters()).device
        tokens = self.tokenizer(list(texts)).to(device)

        with torch.no_grad():
            features = self.model.encode_text(tokens)
            features = features / features.norm(dim=-1, keepdim=True)

        return features

In [ ]:
class SwinV2RemoteCLIPFusion(nn.Module):
    def __init__(
        self,
        image_model_name,
        num_outputs=len(TARGET_COLS),
        text_scale=CONFIG["text_scale"]
    ):
        super().__init__()

        self.text_scale = text_scale

        self.image_backbone = timm.create_model(
            image_model_name,
            pretrained=True,
            num_classes=0
        )

        image_dim = self.image_backbone.num_features

        self.text_encoder = RemoteCLIPTextEncoder(model_name="ViT-B-32")

        with torch.no_grad():
            dummy_text = ["satellite image of land cover"]
            dummy_features = self.text_encoder(dummy_text)
            text_dim = dummy_features.shape[1]

        self.text_proj = nn.Linear(text_dim, image_dim)

        self.gate_layer = nn.Sequential(
            nn.Linear(image_dim + image_dim, image_dim),
            nn.Sigmoid()
        )

        hidden = CONFIG["regressor_hidden_dim"]
        self.regressor = nn.Sequential(
            nn.LayerNorm(image_dim),
            nn.Linear(image_dim, hidden),
            nn.ReLU(),
            nn.Dropout(CONFIG["regressor_dropout"]),
            nn.Linear(hidden, num_outputs)
        )

    def forward(self, image, texts):
        image_features = self.image_backbone(image)

        text_features = self.text_encoder(texts)
        text_features = text_features.to(image_features.device)
        text_features = self.text_proj(text_features)

        gate_input = torch.cat([image_features, text_features], dim=1)
        gate = self.gate_layer(gate_input)

        fused = image_features + self.text_scale * gate * text_features

        return self.regressor(fused)

## Evaluation Functions

This section defines evaluation procedures for both image only and multimodal models.

Mean Absolute Error (MAE) is used as the primary evaluation metric. Root Mean Squared Error (RMSE) is also computed for additional analysis. Furthermore, Class-wise MAE is calculated to analyze performance across land-cover categories

These metrics provide both overall and class-level performance insights.

In [ ]:
@torch.no_grad()
def evaluate_model(model, loader, config, model_type):
    """
    Unified evaluation for all model types.
    model_type: "image" | "text" | "remoteclip"
    """
    model.eval()

    criterion = nn.L1Loss()
    total_loss = 0
    all_preds = []
    all_targets = []

    for batch in loader:
        images  = batch["image"].to(config["device"])
        targets = batch["target"].to(config["device"])

        if model_type == "image":
            preds = model(images)
        elif model_type == "text":
            input_ids      = batch["input_ids"].to(config["device"])
            attention_mask = batch["attention_mask"].to(config["device"])
            preds = model(images, input_ids, attention_mask)
        elif model_type == "remoteclip":
            preds = model(images, batch["text"])

        loss = criterion(preds, targets)
        total_loss += loss.item() * images.size(0)
        all_preds.append(preds.cpu().numpy())
        all_targets.append(targets.cpu().numpy())

    y_pred = np.vstack(all_preds)
    y_true = np.vstack(all_targets)

    mae       = mean_absolute_error(y_true, y_pred)
    rmse      = mean_squared_error(y_true, y_pred) ** 0.5
    class_mae = np.mean(np.abs(y_true - y_pred), axis=0)

    return total_loss / len(loader.dataset), mae, rmse, class_mae, y_pred, y_true

## Training: Image Only Model

This function defines the training loop for the image only baseline. The loss function is Mean Absolute Error (L1 loss). The AdamW optimizer with weight decay is used.

Training progress is logged using Weights & Biases (W&B).

## Training: Multimodal Model

This function extends the training process to multimodal inputs.

Differences from image only training is that the image and tokenized text inputs are used. Moreover, the multimodal forward pass combines features using gated fusion.

The same training configuration is maintained. This ensures a fair comparison between image only and multimodal models.

In [ ]:
def train_model(model, train_loader, val_loader, config, model_type):
    """
    Unified training loop for all model types.
    model_type: "image" | "text" | "remoteclip"
    """
    criterion = nn.L1Loss()
    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=config["lr"],
        weight_decay=config["weight_decay"]
    )

    best_val_mae    = float("inf")
    best_state      = None
    patience_counter = 0
    history         = []

    for epoch in range(config["epochs"]):
        model.train()
        train_loss = 0

        for batch in train_loader:
            optimizer.zero_grad()

            images  = batch["image"].to(config["device"])
            targets = batch["target"].to(config["device"])

            if model_type == "image":
                preds = model(images)
            elif model_type == "text":
                input_ids      = batch["input_ids"].to(config["device"])
                attention_mask = batch["attention_mask"].to(config["device"])
                preds = model(images, input_ids, attention_mask)
            elif model_type == "remoteclip":
                preds = model(images, batch["text"])

            loss = criterion(preds, targets)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), config["grad_clip"])
            optimizer.step()

            train_loss += loss.item() * images.size(0)

        train_loss /= len(train_loader.dataset)

        val_loss, val_mae, val_rmse, _, _, _ = evaluate_model(
            model, val_loader, config, model_type
        )

        history.append({
            "epoch":      epoch + 1,
            "train_loss": train_loss,
            "val_loss":   val_loss,
            "val_mae":    val_mae,
            "val_rmse":   val_rmse
        })

        print(
            f"Epoch {epoch+1}/{config['epochs']} | "
            f"Train Loss: {train_loss:.4f} | "
            f"Val MAE: {val_mae:.4f} | "
            f"Val RMSE: {val_rmse:.4f}"
        )

        wandb.log({
            "epoch":      epoch + 1,
            "train_loss": train_loss,
            "val_loss":   val_loss,
            "val_mae":    val_mae,
            "val_rmse":   val_rmse
        })

        if val_mae < best_val_mae:
            best_val_mae     = val_mae
            best_state       = model.state_dict()
            patience_counter = 0
        else:
            patience_counter += 1

        if patience_counter >= config["early_stopping_patience"]:
            print("Early stopping triggered.")
            break

    model.load_state_dict(best_state)
    return model, pd.DataFrame(history), best_val_mae

## Experiment: Image Only Baseline

This function runs the image only experiment using the SwinV2 backbone.

Metrics (MAE, RMSE, and class-wise MAE) are logged using Weights & Biases (W&B).

This experiment serves as the baseline for comparing multimodal configurations.

In [ ]:
def run_swin_image_only(
    wandb_project=CONFIG["wandb_project"],
    save_predictions=False
):
    seed_everything(CONFIG["seed"])

    run_config = {
        **CONFIG,
        "experiment":   "swinv2_image_only",
        "fusion":       "none",
        "text_column":  "none",
        "text_model":   "none",
    }

    wandb.init(
        project=wandb_project,
        name="swinv2_image_only",
        config=run_config,
        reinit=True
    )

    train_ds = LandCoverDataset(train_df, IMAGE_DIR, transform=train_tfms)
    val_ds   = LandCoverDataset(val_df,   IMAGE_DIR, transform=eval_tfms)
    test_ds  = LandCoverDataset(test_df,  IMAGE_DIR, transform=eval_tfms)

    train_loader = DataLoader(train_ds, batch_size=CONFIG["batch_size"], shuffle=True,  num_workers=CONFIG["num_workers"], pin_memory=True)
    val_loader   = DataLoader(val_ds,   batch_size=CONFIG["batch_size"], shuffle=False, num_workers=CONFIG["num_workers"], pin_memory=True)
    test_loader  = DataLoader(test_ds,  batch_size=CONFIG["batch_size"], shuffle=False, num_workers=CONFIG["num_workers"], pin_memory=True)

    model = ImageOnlyModel(
        model_name=CONFIG["model_name"],
        num_outputs=len(TARGET_COLS)
    ).to(CONFIG["device"])

    model, history, best_val_mae = train_model(model, train_loader, val_loader, CONFIG, model_type="image")

    test_loss, test_mae, test_rmse, class_mae, y_pred, y_true = evaluate_model(
        model, test_loader, CONFIG, model_type="image"
    )

    if save_predictions:
        pred_path = os.path.join(
            CONFIG["predictions_dir"],
            f"predictions_{CONFIG['model_name']}_image_only.csv"
        )
        pd.DataFrame({
            "filename": test_df["filename"].values,
            **{f"pred_{cls}": y_pred[:, i] for i, cls in enumerate(TARGET_COLS)},
            **{f"true_{cls}": y_true[:, i] for i, cls in enumerate(TARGET_COLS)}
        }).to_csv(pred_path, index=False)

    wandb.log({"test_loss": test_loss, "test_mae": test_mae, "test_rmse": test_rmse})
    wandb.log({
        "class_mae_table": wandb.Table(
            dataframe=pd.DataFrame({"class": TARGET_COLS, "class_mae": class_mae})
        )
    })
    wandb.finish()

    return {
        "experiment": "swinv2_image_only",
        "text_model": "none",
        "test_mae":   test_mae,
        "test_rmse":  test_rmse,
        **{f"mae_{cls}": class_mae[i] for i, cls in enumerate(TARGET_COLS)}
    }

## Experiment: Multimodal (Image + Text)

This function runs multimodal experiments by combining image features with textual information.

Different caption sources and text encoders (MiniLM, DistilBERT) are utilized

For each configuration the model is initialized with a gated fusion mechanism, text contribution is controlled via a scaling factor and the same training and evaluation pipeline is applied

Results are logged to W&B, enabling comparison across different text sources and encoders. This setup allows systematic evaluation of how textual information impacts model performance.

In [ ]:
def run_swin_transformer_text(
    text_col,
    text_model,
    text_scale=CONFIG["text_scale"],
    wandb_project=CONFIG["wandb_project"],
    save_predictions=False
):
    seed_everything(CONFIG["seed"])

    run_config = {
        **CONFIG,
        "experiment":   "swinv2_transformer_text",
        "fusion":       "gated_frozen_scaled",
        "text_column":  text_col,
        "text_model":   text_model,
        "text_scale":   text_scale,
    }

    run_name = f"swinv2_{text_col}_{text_model.split('/')[-1]}_scale_{text_scale}"

    wandb.init(project=wandb_project, name=run_name, config=run_config, reinit=True)

    tokenizer = AutoTokenizer.from_pretrained(text_model)

    train_ds = LandCoverDataset(train_df, IMAGE_DIR, transform=train_tfms, tokenizer=tokenizer, text_col=text_col)
    val_ds   = LandCoverDataset(val_df,   IMAGE_DIR, transform=eval_tfms,  tokenizer=tokenizer, text_col=text_col)
    test_ds  = LandCoverDataset(test_df,  IMAGE_DIR, transform=eval_tfms,  tokenizer=tokenizer, text_col=text_col)

    train_loader = DataLoader(train_ds, batch_size=CONFIG["batch_size"], shuffle=True,  num_workers=CONFIG["num_workers"], pin_memory=True)
    val_loader   = DataLoader(val_ds,   batch_size=CONFIG["batch_size"], shuffle=False, num_workers=CONFIG["num_workers"], pin_memory=True)
    test_loader  = DataLoader(test_ds,  batch_size=CONFIG["batch_size"], shuffle=False, num_workers=CONFIG["num_workers"], pin_memory=True)

    model = ImageTextGatedFrozenScaledModel(
        image_model_name=CONFIG["model_name"],
        text_model_name=text_model,
        num_outputs=len(TARGET_COLS),
        text_scale=text_scale
    ).to(CONFIG["device"])

    model, history, best_val_mae = train_model(model, train_loader, val_loader, CONFIG, model_type="text")

    test_loss, test_mae, test_rmse, class_mae, y_pred, y_true = evaluate_model(
        model, test_loader, CONFIG, model_type="text"
    )

    if save_predictions:
        pred_path = os.path.join(
            CONFIG["predictions_dir"],
            f"predictions_{CONFIG['model_name']}_{text_col}_{text_model.split('/')[-1]}_scale_{text_scale}.csv"
        )
        pd.DataFrame({
            "filename": test_df["filename"].values,
            **{f"pred_{cls}": y_pred[:, i] for i, cls in enumerate(TARGET_COLS)},
            **{f"true_{cls}": y_true[:, i] for i, cls in enumerate(TARGET_COLS)}
        }).to_csv(pred_path, index=False)

    wandb.log({"test_loss": test_loss, "test_mae": test_mae, "test_rmse": test_rmse, "best_val_mae": best_val_mae})
    wandb.log({
        "class_mae_table": wandb.Table(
            dataframe=pd.DataFrame({"class": TARGET_COLS, "class_mae": class_mae})
        )
    })
    wandb.finish()

    return {
        "experiment":  run_name,
        "text_column": text_col,
        "text_model":  text_model,
        "text_scale":  text_scale,
        "val_mae":     best_val_mae,
        "test_mae":    test_mae,
        "test_rmse":   test_rmse,
        "class_mae":   class_mae,
        **{f"mae_{cls}": class_mae[i] for i, cls in enumerate(TARGET_COLS)}
    }

In [ ]:
def run_swinv2_remoteclip_text(
    text_col,
    text_scale=CONFIG["text_scale"],
    wandb_project=CONFIG["wandb_project"],
    save_predictions=False
):
    seed_everything(CONFIG["seed"])

    run_config = {
        **CONFIG,
        "experiment":  "swinv2_remoteclip_text",
        "fusion":      "gated_frozen_scaled",
        "text_column": text_col,
        "text_model":  "RemoteCLIP ViT-B-32",
        "text_scale":  text_scale,
    }

    run_name = f"swinv2_remoteclip_{text_col}_scale_{text_scale}"

    wandb.init(project=wandb_project, name=run_name, config=run_config, reinit=True)

    train_ds = LandCoverRawTextDataset(train_df, IMAGE_DIR, transform=train_tfms, text_col=text_col)
    val_ds   = LandCoverRawTextDataset(val_df,   IMAGE_DIR, transform=eval_tfms,  text_col=text_col)
    test_ds  = LandCoverRawTextDataset(test_df,  IMAGE_DIR, transform=eval_tfms,  text_col=text_col)

    train_loader = DataLoader(train_ds, batch_size=CONFIG["batch_size"], shuffle=True,  num_workers=CONFIG["num_workers"], pin_memory=True)
    val_loader   = DataLoader(val_ds,   batch_size=CONFIG["batch_size"], shuffle=False, num_workers=CONFIG["num_workers"], pin_memory=True)
    test_loader  = DataLoader(test_ds,  batch_size=CONFIG["batch_size"], shuffle=False, num_workers=CONFIG["num_workers"], pin_memory=True)

    model = SwinV2TinyRemoteCLIPFusion(
        image_model_name=CONFIG["model_name"],
        num_outputs=len(TARGET_COLS),
        text_scale=text_scale
    ).to(CONFIG["device"])

    model, history, best_val_mae = train_model(model, train_loader, val_loader, CONFIG, model_type="remoteclip")

    test_loss, test_mae, test_rmse, class_mae, y_pred, y_true = evaluate_model(
        model, test_loader, CONFIG, model_type="remoteclip"
    )

    if save_predictions:
        pred_path = os.path.join(
            CONFIG["predictions_dir"],
            f"predictions_{CONFIG['model_name']}_{text_col}_remoteclip_scale_{text_scale}.csv"
        )
        pd.DataFrame({
            "filename": test_df["filename"].values,
            **{f"pred_{cls}": y_pred[:, i] for i, cls in enumerate(TARGET_COLS)},
            **{f"true_{cls}": y_true[:, i] for i, cls in enumerate(TARGET_COLS)}
        }).to_csv(pred_path, index=False)

    wandb.log({"test_loss": test_loss, "test_mae": test_mae, "test_rmse": test_rmse, "best_val_mae": best_val_mae})
    wandb.log({
        "class_mae_table": wandb.Table(
            dataframe=pd.DataFrame({"class": TARGET_COLS, "class_mae": class_mae})
        )
    })
    wandb.finish()

    return {
        "experiment":  run_name,
        "text_column": text_col,
        "text_model":  "RemoteCLIP ViT-B-32",
        "text_scale":  text_scale,
        "val_mae":     best_val_mae,
        "test_mae":    test_mae,
        "test_rmse":   test_rmse,
        "class_mae":   class_mae,
        **{f"mae_{cls}": class_mae[i] for i, cls in enumerate(TARGET_COLS)}
    }

# Running Experiments

In [ ]:
wandb.login()

/usr/local/lib/python3.12/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)
wandb: (1) Create a W&B account
wandb: (2) Use an existing W&B account
wandb: (3) Don't visualize my results


wandb: Enter your choice: 2


wandb: You chose 'Use an existing W&B account'
wandb: Logging into https://api.wandb.ai. (Learn how to deploy a W&B server locally: https://wandb.me/wandb-server)
wandb: Create a new API key at: https://wandb.ai/authorize?ref=models
wandb: Store your API key securely and do not share it.


wandb: Paste your API key and hit enter: ··········


wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: ayseatci00 (ayseatci00-) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


True

In [ ]:
results = []

## Image Only Experiment

In [ ]:
results.append(run_swin_image_only(save_predictions=True))
pd.DataFrame(results)

wandb: WARNING Using a boolean value for 'reinit' is deprecated. Use 'return_previous' or 'finish_previous' instead.


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


model.safetensors:   0%|          | 0.00/115M [00:00<?, ?B/s]

Epoch 1/15 | Train Loss: 10.9368 | Val MAE: 7.6370 | Val RMSE: 17.5412
Epoch 2/15 | Train Loss: 6.6776 | Val MAE: 4.7681 | Val RMSE: 11.2401
Epoch 3/15 | Train Loss: 5.0621 | Val MAE: 3.8214 | Val RMSE: 9.8392
Epoch 4/15 | Train Loss: 4.3678 | Val MAE: 3.0987 | Val RMSE: 7.9711
Epoch 5/15 | Train Loss: 4.0015 | Val MAE: 2.9661 | Val RMSE: 7.7742
Epoch 6/15 | Train Loss: 3.7039 | Val MAE: 3.0301 | Val RMSE: 7.9744
Epoch 7/15 | Train Loss: 3.4473 | Val MAE: 2.7813 | Val RMSE: 7.3366
Epoch 8/15 | Train Loss: 3.2905 | Val MAE: 2.7061 | Val RMSE: 7.4839
Epoch 9/15 | Train Loss: 3.0926 | Val MAE: 2.4959 | Val RMSE: 6.7511
Epoch 10/15 | Train Loss: 2.9384 | Val MAE: 2.6021 | Val RMSE: 6.4188
Epoch 11/15 | Train Loss: 2.8589 | Val MAE: 2.3889 | Val RMSE: 6.3558
Epoch 12/15 | Train Loss: 2.7165 | Val MAE: 2.3664 | Val RMSE: 6.2739
Epoch 13/15 | Train Loss: 2.6055 | Val MAE: 2.4033 | Val RMSE: 6.3176
Epoch 14/15 | Train Loss: 2.5402 | Val MAE: 2.2371 | Val RMSE: 5.7950
Epoch 15/15 | Train Loss: 

epoch,▁▁▂▃▃▃▄▅▅▅▆▇▇▇█
test_loss,▁
test_mae,▁
test_rmse,▁
train_loss,█▄▃▃▂▂▂▂▂▁▁▁▁▁▁
val_loss,█▄▃▂▂▂▂▂▁▁▁▁▁▁▁
val_mae,█▄▃▂▂▂▂▂▁▁▁▁▁▁▁
val_rmse,█▄▃▂▂▂▂▂▂▁▁▁▁▁▁
epoch,15
test_loss,2.48358
test_mae,2.48358


,experiment,test_mae,test_rmse,mae_Tree,mae_Shrub,mae_Grass,mae_Crop,mae_Built-up,mae_Barren,mae_Water
0,swinv2_image_only,2.483577,6.487426,2.720171,0.968663,6.086374,4.688375,0.519106,2.045412,0.356938


## MiniLM Experiments

In [ ]:
for text_col in TEXT_COLS_CLEAN:
    results.append(
        run_swin_transformer_text(
            text_col=text_col,
            text_model="sentence-transformers/all-MiniLM-L6-v2",
            text_scale=CONFIG["text_scale"],
            save_predictions=True
        )
    )

pd.DataFrame(results).sort_values("test_mae")

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

...

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Epoch 1/15 | Train Loss: 10.7094 | Val MAE: 7.3791 | Val RMSE: 16.6857
Epoch 2/15 | Train Loss: 6.4629 | Val MAE: 4.4022 | Val RMSE: 10.5624
Epoch 3/15 | Train Loss: 4.8936 | Val MAE: 3.4858 | Val RMSE: 8.9770
Epoch 4/15 | Train Loss: 4.2818 | Val MAE: 3.1546 | Val RMSE: 8.0113
Epoch 5/15 | Train Loss: 3.9376 | Val MAE: 3.0544 | Val RMSE: 7.9458
Epoch 6/15 | Train Loss: 3.7625 | Val MAE: 2.8475 | Val RMSE: 7.5636
Epoch 7/15 | Train Loss: 3.4928 | Val MAE: 3.0490 | Val RMSE: 8.0967
Epoch 8/15 | Train Loss: 3.2803 | Val MAE: 2.5948 | Val RMSE: 6.6936
Epoch 9/15 | Train Loss: 3.0650 | Val MAE: 2.4747 | Val RMSE: 6.3930
Epoch 10/15 | Train Loss: 2.9359 | Val MAE: 2.5469 | Val RMSE: 6.3233
Epoch 11/15 | Train Loss: 2.8528 | Val MAE: 2.6038 | Val RMSE: 6.3704
Epoch 12/15 | Train Loss: 2.6575 | Val MAE: 2.3221 | Val RMSE: 5.9854
Epoch 13/15 | Train Loss: 2.5484 | Val MAE: 2.2083 | Val RMSE: 5.9833
Epoch 14/15 | Train Loss: 2.5062 | Val MAE: 2.3680 | Val RMSE: 6.0458
Epoch 15/15 | Train Loss: 

best_val_mae,▁
epoch,▁▁▂▃▃▃▄▅▅▅▆▇▇▇█
test_loss,▁
test_mae,▁
test_rmse,▁
train_loss,█▄▃▃▂▂▂▂▂▁▁▁▁▁▁
val_loss,█▄▃▂▂▂▂▂▁▁▂▁▁▁▁
val_mae,█▄▃▂▂▂▂▂▁▁▂▁▁▁▁
val_rmse,█▄▃▂▂▂▂▁▁▁▁▁▁▁▁
best_val_mae,2.18793
epoch,15


,experiment,test_mae,test_rmse,mae_Tree,mae_Shrub,mae_Grass,mae_Crop,mae_Built-up,mae_Barren,mae_Water,text_column,text_model,text_scale,val_mae,class_mae
2,swinv2_text_qwen3_4b_clean_all-MiniLM-L6-v2_sc...,2.044218,4.662678,2.501036,0.955026,5.604757,2.769237,0.457257,1.707687,0.314527,text_qwen3_4b_clean,sentence-transformers/all-MiniLM-L6-v2,0.7,1.813002,"[2.5010364, 0.9550259, 5.6047573, 2.7692366, 0..."
1,swinv2_hybrid_gemma3_4b_clean_all-MiniLM-L6-v2...,2.078193,5.153962,2.384802,0.957509,5.129480,3.499064,0.468410,1.759308,0.348780,hybrid_gemma3_4b_clean,sentence-transformers/all-MiniLM-L6-v2,0.7,2.000597,"[2.384802, 0.9575093, 5.1294804, 3.4990637, 0...."
3,swinv2_vision_gemma3_4b_clean_all-MiniLM-L6-v2...,2.360168,6.202637,2.470267,0.958383,5.953213,4.325037,0.441480,2.031436,0.341362,vision_gemma3_4b_clean,sentence-transformers/all-MiniLM-L6-v2,0.7,2.187932,"[2.4702673, 0.958383, 5.953213, 4.325037, 0.44..."
0,swinv2_image_only,2.483577,6.487426,2.720171,0.968663,6.086374,4.688375,0.519106,2.045412,0.356938,NaN,NaN,NaN,NaN,NaN


## Remote CLIP Experiments

In [ ]:
for text_col in TEXT_COLS_CLEAN:
    results.append(
        run_swinv2_remoteclip_text(
            text_col=text_col,
            text_scale=CONFIG["text_scale"],
            save_predictions=True
        )
    )

pd.DataFrame(results).sort_values("test_mae")

RemoteCLIP-ViT-B-32.pt:   0%|          | 0.00/605M [00:00<?, ?B/s]

RemoteCLIP weights loaded
Epoch 1/15 | Train Loss: 10.9920 | Val MAE: 7.7242 | Val RMSE: 17.5976
Epoch 2/15 | Train Loss: 6.5418 | Val MAE: 4.3402 | Val RMSE: 10.5825
Epoch 3/15 | Train Loss: 4.8936 | Val MAE: 3.4855 | Val RMSE: 9.0045
Epoch 4/15 | Train Loss: 4.1849 | Val MAE: 3.2462 | Val RMSE: 8.5186
Epoch 5/15 | Train Loss: 3.8999 | Val MAE: 2.8638 | Val RMSE: 7.4988
Epoch 6/15 | Train Loss: 3.5770 | Val MAE: 2.6230 | Val RMSE: 6.9699
Epoch 7/15 | Train Loss: 3.2990 | Val MAE: 2.4348 | Val RMSE: 6.3245
Epoch 8/15 | Train Loss: 3.0520 | Val MAE: 2.3501 | Val RMSE: 5.9111
Epoch 9/15 | Train Loss: 2.8417 | Val MAE: 2.1802 | Val RMSE: 5.7344
Epoch 10/15 | Train Loss: 2.6899 | Val MAE: 2.0996 | Val RMSE: 5.1622
Epoch 11/15 | Train Loss: 2.5284 | Val MAE: 2.2201 | Val RMSE: 5.2450
Epoch 12/15 | Train Loss: 2.3946 | Val MAE: 2.0473 | Val RMSE: 5.2005
Epoch 13/15 | Train Loss: 2.2809 | Val MAE: 1.9213 | Val RMSE: 4.6467
Epoch 14/15 | Train Loss: 2.2518 | Val MAE: 1.8435 | Val RMSE: 4.5854


best_val_mae,▁
epoch,▁▁▂▃▃▃▄▅▅▅▆▇▇▇█
test_loss,▁
test_mae,▁
test_rmse,▁
train_loss,█▄▃▃▂▂▂▂▂▁▁▁▁▁▁
val_loss,█▄▃▃▂▂▂▂▁▁▁▁▁▁▁
val_mae,█▄▃▃▂▂▂▂▁▁▁▁▁▁▁
val_rmse,█▄▃▃▃▂▂▂▂▁▁▁▁▁▁
best_val_mae,1.84348
epoch,15


RemoteCLIP weights loaded
Epoch 1/15 | Train Loss: 10.9293 | Val MAE: 7.4866 | Val RMSE: 17.1415
Epoch 2/15 | Train Loss: 6.4828 | Val MAE: 4.4952 | Val RMSE: 10.7045
Epoch 3/15 | Train Loss: 4.9046 | Val MAE: 3.3290 | Val RMSE: 8.6152
Epoch 4/15 | Train Loss: 4.1789 | Val MAE: 3.5777 | Val RMSE: 9.0059
Epoch 5/15 | Train Loss: 3.8475 | Val MAE: 2.7895 | Val RMSE: 7.3530
Epoch 6/15 | Train Loss: 3.4676 | Val MAE: 2.8818 | Val RMSE: 7.0988
Epoch 7/15 | Train Loss: 3.1743 | Val MAE: 2.4756 | Val RMSE: 6.3027
Epoch 8/15 | Train Loss: 2.9954 | Val MAE: 2.3600 | Val RMSE: 5.8250
Epoch 9/15 | Train Loss: 2.7982 | Val MAE: 2.2268 | Val RMSE: 5.4522
Epoch 10/15 | Train Loss: 2.6279 | Val MAE: 2.0703 | Val RMSE: 4.9750
Epoch 11/15 | Train Loss: 2.4965 | Val MAE: 2.0234 | Val RMSE: 4.8445
Epoch 12/15 | Train Loss: 2.3665 | Val MAE: 2.0132 | Val RMSE: 4.7335
Epoch 13/15 | Train Loss: 2.2528 | Val MAE: 1.9533 | Val RMSE: 4.5390
Epoch 14/15 | Train Loss: 2.1884 | Val MAE: 1.7879 | Val RMSE: 4.2850


best_val_mae,▁
epoch,▁▁▂▃▃▃▄▅▅▅▆▇▇▇█
test_loss,▁
test_mae,▁
test_rmse,▁
train_loss,█▄▃▃▂▂▂▂▂▁▁▁▁▁▁
val_loss,█▄▃▃▂▂▂▂▂▁▁▁▁▁▁
val_mae,█▄▃▃▂▂▂▂▂▁▁▁▁▁▁
val_rmse,█▄▃▄▃▃▂▂▂▁▁▁▁▁▁
best_val_mae,1.78788
epoch,15


RemoteCLIP weights loaded
Epoch 1/15 | Train Loss: 10.9471 | Val MAE: 7.6488 | Val RMSE: 17.4064
Epoch 2/15 | Train Loss: 6.4982 | Val MAE: 4.5902 | Val RMSE: 11.0054
Epoch 3/15 | Train Loss: 4.9354 | Val MAE: 3.4183 | Val RMSE: 8.7954
Epoch 4/15 | Train Loss: 4.3185 | Val MAE: 3.4460 | Val RMSE: 9.2555
Epoch 5/15 | Train Loss: 3.9395 | Val MAE: 2.9643 | Val RMSE: 7.9465
Epoch 6/15 | Train Loss: 3.7018 | Val MAE: 2.9598 | Val RMSE: 7.4405
Epoch 7/15 | Train Loss: 3.4914 | Val MAE: 2.5589 | Val RMSE: 6.8682
Epoch 8/15 | Train Loss: 3.2677 | Val MAE: 2.5982 | Val RMSE: 6.6781
Epoch 9/15 | Train Loss: 3.0946 | Val MAE: 2.3684 | Val RMSE: 6.3527
Epoch 10/15 | Train Loss: 2.8786 | Val MAE: 2.4907 | Val RMSE: 6.2154
Epoch 11/15 | Train Loss: 2.7636 | Val MAE: 2.2991 | Val RMSE: 6.0130
Epoch 12/15 | Train Loss: 2.6274 | Val MAE: 2.2433 | Val RMSE: 5.9061
Epoch 13/15 | Train Loss: 2.5199 | Val MAE: 2.2386 | Val RMSE: 5.9467
Epoch 14/15 | Train Loss: 2.4850 | Val MAE: 2.1672 | Val RMSE: 5.6565


best_val_mae,▁
epoch,▁▁▂▃▃▃▄▅▅▅▆▇▇▇█
test_loss,▁
test_mae,▁
test_rmse,▁
train_loss,█▄▃▃▂▂▂▂▂▁▁▁▁▁▁
val_loss,█▄▃▃▂▂▂▂▁▁▁▁▁▁▁
val_mae,█▄▃▃▂▂▂▂▁▁▁▁▁▁▁
val_rmse,█▄▃▃▂▂▂▂▁▁▁▁▁▁▁
best_val_mae,2.16715
epoch,15


,experiment,test_mae,test_rmse,mae_Tree,mae_Shrub,mae_Grass,mae_Crop,mae_Built-up,mae_Barren,mae_Water,text_column,text_model,text_scale,val_mae,class_mae
2,swinv2_text_qwen3_4b_clean_all-MiniLM-L6-v2_sc...,2.044218,4.662678,2.501036,0.955026,5.604757,2.769237,0.457257,1.707687,0.314527,text_qwen3_4b_clean,sentence-transformers/all-MiniLM-L6-v2,0.7,1.813002,"[2.5010364, 0.9550259, 5.6047573, 2.7692366, 0..."
1,swinv2_hybrid_gemma3_4b_clean_all-MiniLM-L6-v2...,2.078193,5.153962,2.384802,0.957509,5.129480,3.499064,0.468410,1.759308,0.348780,hybrid_gemma3_4b_clean,sentence-transformers/all-MiniLM-L6-v2,0.7,2.000597,"[2.384802, 0.9575093, 5.1294804, 3.4990637, 0...."
5,swinv2_remoteclip_text_qwen3_4b_clean_scale_0.7,2.095319,4.763024,2.960334,0.960769,5.227705,2.922526,0.422440,1.772410,0.401049,text_qwen3_4b_clean,RemoteCLIP ViT-B-32,0.7,1.787881,"[2.9603343, 0.9607688, 5.2277045, 2.9225264, 0..."
4,swinv2_remoteclip_hybrid_gemma3_4b_clean_scale...,2.112155,4.901970,3.168965,0.961131,4.958211,2.953909,0.407046,2.005704,0.330122,hybrid_gemma3_4b_clean,RemoteCLIP ViT-B-32,0.7,1.843484,"[3.1689649, 0.9611308, 4.9582114, 2.953909, 0...."
3,swinv2_vision_gemma3_4b_clean_all-MiniLM-L6-v2...,2.360168,6.202637,2.470267,0.958383,5.953213,4.325037,0.441480,2.031436,0.341362,vision_gemma3_4b_clean,sentence-transformers/all-MiniLM-L6-v2,0.7,2.187932,"[2.4702673, 0.958383, 5.953213, 4.325037, 0.44..."
6,swinv2_remoteclip_vision_gemma3_4b_clean_scale...,2.408850,6.235519,3.034881,0.962156,5.982058,4.163283,0.438985,1.904258,0.376331,vision_gemma3_4b_clean,RemoteCLIP ViT-B-32,0.7,2.167152,"[3.0348809, 0.9621565, 5.982058, 4.163283, 0.4..."
0,swinv2_image_only,2.483577,6.487426,2.720171,0.968663,6.086374,4.688375,0.519106,2.045412,0.356938,NaN,NaN,NaN,NaN,NaN


## DistilBERT Experiments

In [ ]:
for text_col in TEXT_COLS_CLEAN:
    results.append(
        run_swin_transformer_text(
            text_col=text_col,
            text_model="distilbert-base-uncased",
            text_scale=CONFIG["text_scale"],
            save_predictions=True
        )
    )

pd.DataFrame(results).sort_values("test_mae")

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

...

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertModel LOAD REPORT from: distilbert-base-uncased
Key                     | Status     |  | 
------------------------+------------+--+-
vocab_layer_norm.bias   | UNEXPECTED |  | 
vocab_transform.bias    | UNEXPECTED |  | 
vocab_projector.bias    | UNEXPECTED |  | 
vocab_transform.weight  | UNEXPECTED |  | 
vocab_layer_norm.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Epoch 1/15 | Train Loss: 10.9691 | Val MAE: 7.7475 | Val RMSE: 17.3927
Epoch 2/15 | Train Loss: 6.4752 | Val MAE: 4.4049 | Val RMSE: 10.7153
Epoch 3/15 | Train Loss: 4.9450 | Val MAE: 3.5285 | Val RMSE: 8.8567
Epoch 4/15 | Train Loss: 4.2782 | Val MAE: 3.0387 | Val RMSE: 8.0155
Epoch 5/15 | Train Loss: 3.9176 | Val MAE: 3.0803 | Val RMSE: 8.3555
Epoch 6/15 | Train Loss: 3.7362 | Val MAE: 2.8962 | Val RMSE: 7.6675
Epoch 7/15 | Train Loss: 3.4412 | Val MAE: 3.1224 | Val RMSE: 7.8965
Epoch 8/15 | Train Loss: 3.2613 | Val MAE: 2.7916 | Val RMSE: 7.3393
Epoch 9/15 | Train Loss: 3.0343 | Val MAE: 2.4679 | Val RMSE: 6.7105
Epoch 10/15 | Train Loss: 2.8516 | Val MAE: 2.5954 | Val RMSE: 6.4775
Epoch 11/15 | Train Loss: 2.7618 | Val MAE: 2.5287 | Val RMSE: 7.0283
Epoch 12/15 | Train Loss: 2.6417 | Val MAE: 2.4358 | Val RMSE: 6.4015
Epoch 13/15 | Train Loss: 2.5582 | Val MAE: 2.3581 | Val RMSE: 6.1562
Epoch 14/15 | Train Loss: 2.4610 | Val MAE: 2.4068 | Val RMSE: 6.4877
Epoch 15/15 | Train Loss: 

best_val_mae,▁
epoch,▁▁▂▃▃▃▄▅▅▅▆▇▇▇█
test_loss,▁
test_mae,▁
test_rmse,▁
train_loss,█▄▃▃▂▂▂▂▂▁▁▁▁▁▁
val_loss,█▄▃▂▂▂▂▂▁▂▁▁▁▁▁
val_mae,█▄▃▂▂▂▂▂▁▂▁▁▁▁▁
val_rmse,█▄▃▂▃▂▂▂▂▁▂▁▁▁▁
best_val_mae,2.14346
epoch,15


,experiment,test_mae,test_rmse,mae_Tree,mae_Shrub,mae_Grass,mae_Crop,mae_Built-up,mae_Barren,mae_Water,text_column,text_model,text_scale,val_mae,class_mae
8,swinv2_text_qwen3_4b_clean_distilbert-base-unc...,1.878618,4.518867,2.569227,0.950257,4.145995,2.639066,0.429315,2.022730,0.393734,text_qwen3_4b_clean,distilbert-base-uncased,0.7,1.757897,"[2.569227, 0.9502565, 4.1459947, 2.6390662, 0...."
2,swinv2_text_qwen3_4b_clean_all-MiniLM-L6-v2_sc...,2.044218,4.662678,2.501036,0.955026,5.604757,2.769237,0.457257,1.707687,0.314527,text_qwen3_4b_clean,sentence-transformers/all-MiniLM-L6-v2,0.7,1.813002,"[2.5010364, 0.9550259, 5.6047573, 2.7692366, 0..."
1,swinv2_hybrid_gemma3_4b_clean_all-MiniLM-L6-v2...,2.078193,5.153962,2.384802,0.957509,5.129480,3.499064,0.468410,1.759308,0.348780,hybrid_gemma3_4b_clean,sentence-transformers/all-MiniLM-L6-v2,0.7,2.000597,"[2.384802, 0.9575093, 5.1294804, 3.4990637, 0...."
5,swinv2_remoteclip_text_qwen3_4b_clean_scale_0.7,2.095319,4.763024,2.960334,0.960769,5.227705,2.922526,0.422440,1.772410,0.401049,text_qwen3_4b_clean,RemoteCLIP ViT-B-32,0.7,1.787881,"[2.9603343, 0.9607688, 5.2277045, 2.9225264, 0..."
4,swinv2_remoteclip_hybrid_gemma3_4b_clean_scale...,2.112155,4.901970,3.168965,0.961131,4.958211,2.953909,0.407046,2.005704,0.330122,hybrid_gemma3_4b_clean,RemoteCLIP ViT-B-32,0.7,1.843484,"[3.1689649, 0.9611308, 4.9582114, 2.953909, 0...."
9,swinv2_vision_gemma3_4b_clean_distilbert-base-...,2.327044,6.156766,2.609429,0.951131,5.789022,4.166644,0.474781,1.912253,0.386047,vision_gemma3_4b_clean,distilbert-base-uncased,0.7,2.143463,"[2.6094286, 0.9511311, 5.789022, 4.166644, 0.4..."
3,swinv2_vision_gemma3_4b_clean_all-MiniLM-L6-v2...,2.360168,6.202637,2.470267,0.958383,5.953213,4.325037,0.441480,2.031436,0.341362,vision_gemma3_4b_clean,sentence-transformers/all-MiniLM-L6-v2,0.7,2.187932,"[2.4702673, 0.958383, 5.953213, 4.325037, 0.44..."
7,swinv2_hybrid_gemma3_4b_clean_distilbert-base-...,2.363099,6.085603,3.301959,0.963847,5.456616,3.977801,0.429732,2.066000,0.345736,hybrid_gemma3_4b_clean,distilbert-base-uncased,0.7,2.208799,"[3.301959, 0.96384716, 5.456616, 3.977801, 0.4..."
6,swinv2_remoteclip_vision_gemma3_4b_clean_scale...,2.408850,6.235519,3.034881,0.962156,5.982058,4.163283,0.438985,1.904258,0.376331,vision_gemma3_4b_clean,RemoteCLIP ViT-B-32,0.7,2.167152,"[3.0348809, 0.9621565, 5.982058, 4.163283, 0.4..."
0,swinv2_image_only,2.483577,6.487426,2.720171,0.968663,6.086374,4.688375,0.519106,2.045412,0.356938,NaN,NaN,NaN,NaN,NaN


### SwinV2 Experiment Results Summary

In [ ]:
results_df = pd.DataFrame(results).sort_values("test_mae")
results_df

,experiment,test_mae,test_rmse,mae_Tree,mae_Shrub,mae_Grass,mae_Crop,mae_Built-up,mae_Barren,mae_Water,text_column,text_model,text_scale,val_mae,class_mae
8,swinv2_text_qwen3_4b_clean_distilbert-base-unc...,1.878618,4.518867,2.569227,0.950257,4.145995,2.639066,0.429315,2.022730,0.393734,text_qwen3_4b_clean,distilbert-base-uncased,0.7,1.757897,"[2.569227, 0.9502565, 4.1459947, 2.6390662, 0...."
2,swinv2_text_qwen3_4b_clean_all-MiniLM-L6-v2_sc...,2.044218,4.662678,2.501036,0.955026,5.604757,2.769237,0.457257,1.707687,0.314527,text_qwen3_4b_clean,sentence-transformers/all-MiniLM-L6-v2,0.7,1.813002,"[2.5010364, 0.9550259, 5.6047573, 2.7692366, 0..."
1,swinv2_hybrid_gemma3_4b_clean_all-MiniLM-L6-v2...,2.078193,5.153962,2.384802,0.957509,5.129480,3.499064,0.468410,1.759308,0.348780,hybrid_gemma3_4b_clean,sentence-transformers/all-MiniLM-L6-v2,0.7,2.000597,"[2.384802, 0.9575093, 5.1294804, 3.4990637, 0...."
5,swinv2_remoteclip_text_qwen3_4b_clean_scale_0.7,2.095319,4.763024,2.960334,0.960769,5.227705,2.922526,0.422440,1.772410,0.401049,text_qwen3_4b_clean,RemoteCLIP ViT-B-32,0.7,1.787881,"[2.9603343, 0.9607688, 5.2277045, 2.9225264, 0..."
4,swinv2_remoteclip_hybrid_gemma3_4b_clean_scale...,2.112155,4.901970,3.168965,0.961131,4.958211,2.953909,0.407046,2.005704,0.330122,hybrid_gemma3_4b_clean,RemoteCLIP ViT-B-32,0.7,1.843484,"[3.1689649, 0.9611308, 4.9582114, 2.953909, 0...."
9,swinv2_vision_gemma3_4b_clean_distilbert-base-...,2.327044,6.156766,2.609429,0.951131,5.789022,4.166644,0.474781,1.912253,0.386047,vision_gemma3_4b_clean,distilbert-base-uncased,0.7,2.143463,"[2.6094286, 0.9511311, 5.789022, 4.166644, 0.4..."
3,swinv2_vision_gemma3_4b_clean_all-MiniLM-L6-v2...,2.360168,6.202637,2.470267,0.958383,5.953213,4.325037,0.441480,2.031436,0.341362,vision_gemma3_4b_clean,sentence-transformers/all-MiniLM-L6-v2,0.7,2.187932,"[2.4702673, 0.958383, 5.953213, 4.325037, 0.44..."
7,swinv2_hybrid_gemma3_4b_clean_distilbert-base-...,2.363099,6.085603,3.301959,0.963847,5.456616,3.977801,0.429732,2.066000,0.345736,hybrid_gemma3_4b_clean,distilbert-base-uncased,0.7,2.208799,"[3.301959, 0.96384716, 5.456616, 3.977801, 0.4..."
6,swinv2_remoteclip_vision_gemma3_4b_clean_scale...,2.408850,6.235519,3.034881,0.962156,5.982058,4.163283,0.438985,1.904258,0.376331,vision_gemma3_4b_clean,RemoteCLIP ViT-B-32,0.7,2.167152,"[3.0348809, 0.9621565, 5.982058, 4.163283, 0.4..."
0,swinv2_image_only,2.483577,6.487426,2.720171,0.968663,6.086374,4.688375,0.519106,2.045412,0.356938,NaN,NaN,NaN,NaN,NaN


Experiments are conducted using three caption types (text only, hybrid, and vision based) and three text encoders (MiniLM, RemoteCLIP and DistilBERT). A fixed text scaling factor is applied to control the contribution of textual features in the fusion process.

The image-only baseline achieves a test MAE of 2.46. Incorporating textual information leads to consistent improvements, particularly when using text only captions.

The best performance is obtained with text only captions and DistilBERT (MAE: 1.79), followed by MiniLM (MAE: 2.03), indicating that informative textual descriptions significantly enhance prediction quality.

Hybrid captions provide moderate improvements over the baseline but are less effective than text only captions. In contrast, vision based captions do not improve performance and in some cases slightly degrade it, suggesting limited additional information beyond visual features.

Class-wise results show that improvements are more pronounced for dominant classes such as Grass and Crop, while smaller gains are observed for less represented classes like Water and Built-up.

Overall, the results indicate that textual information is beneficial when it provides complementary semantic content, with the strongest gains coming from high quality text only captions.

## Text Scale Experiments

The best caption × encoder combination for SwinV2 based on
validation MAE at scale 0.7 is selected.

In [ ]:
# Filter out image-only row — selection is over multimodal configs only
multimodal_results = [r for r in results if r.get("text_model") not in [None, "none"]]
results_df_multimodal = pd.DataFrame(multimodal_results)

best_config = results_df_multimodal.loc[results_df_multimodal["val_mae"].idxmin()]

print("Best configuration for SwinV2 (by validation MAE)")
print(f"  Text column : {best_config['text_column']}")
print(f"  Text model  : {best_config['text_model']}")
print(f"  Val MAE     : {best_config['val_mae']:.4f}")
print(f"  Test MAE    : {best_config['test_mae']:.4f}")

BEST_TEXT_COL   = best_config["text_column"]
BEST_TEXT_MODEL = best_config["text_model"]
IS_REMOTECLIP   = (BEST_TEXT_MODEL == "RemoteCLIP ViT-B-32")

Best configuration for SwinV2 (by validation MAE)
  Text column : text_qwen3_4b_clean
  Text model  : distilbert-base-uncased
  Val MAE     : 1.7579
  Test MAE    : 1.8786


## Text Scale Ablation

Using the best configuration selected above, we sweep text_scale across
[0.3, 0.5, 0.7, 1.0, 1.5].

In [ ]:
TEXT_SCALES = [0.3, 0.5, 0.7, 1.0, 1.5]

scale_results = []

for scale in TEXT_SCALES:
    print(f"\nRunning scale={scale} | col={BEST_TEXT_COL} | model={BEST_TEXT_MODEL}")

    if IS_REMOTECLIP:
        result = run_swinv2_remoteclip_text(
            text_col=BEST_TEXT_COL,
            text_scale=scale,
            wandb_project=CONFIG["wandb_project_scale"],
            save_predictions=True
        )
    else:
        result = run_swin_transformer_text(
            text_col=BEST_TEXT_COL,
            text_model=BEST_TEXT_MODEL,
            text_scale=scale,
            wandb_project=CONFIG["wandb_project_scale"],
            save_predictions=True
        )

    scale_results.append(result)

scale_df = pd.DataFrame(scale_results).sort_values("text_scale").reset_index(drop=True)
scale_df


Running scale=0.3 | col=text_qwen3_4b_clean | model=distilbert-base-uncased


Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertModel LOAD REPORT from: distilbert-base-uncased
Key                     | Status     |  | 
------------------------+------------+--+-
vocab_layer_norm.bias   | UNEXPECTED |  | 
vocab_transform.bias    | UNEXPECTED |  | 
vocab_projector.bias    | UNEXPECTED |  | 
vocab_transform.weight  | UNEXPECTED |  | 
vocab_layer_norm.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Epoch 1/15 | Train Loss: 10.9304 | Val MAE: 7.6243 | Val RMSE: 17.2259
Epoch 2/15 | Train Loss: 6.4428 | Val MAE: 4.1965 | Val RMSE: 10.3809
Epoch 3/15 | Train Loss: 4.6626 | Val MAE: 3.2265 | Val RMSE: 8.1373
Epoch 4/15 | Train Loss: 3.9806 | Val MAE: 2.7320 | Val RMSE: 7.2630
Epoch 5/15 | Train Loss: 3.6339 | Val MAE: 2.6315 | Val RMSE: 6.8579
Epoch 6/15 | Train Loss: 3.3861 | Val MAE: 2.4133 | Val RMSE: 6.4189
Epoch 7/15 | Train Loss: 3.1621 | Val MAE: 2.4990 | Val RMSE: 6.1394
Epoch 8/15 | Train Loss: 2.9751 | Val MAE: 2.2075 | Val RMSE: 5.6962
Epoch 9/15 | Train Loss: 2.7705 | Val MAE: 2.0458 | Val RMSE: 5.3457
Epoch 10/15 | Train Loss: 2.6262 | Val MAE: 2.0667 | Val RMSE: 5.1936
Epoch 11/15 | Train Loss: 2.5225 | Val MAE: 2.3148 | Val RMSE: 5.6298
Epoch 12/15 | Train Loss: 2.4152 | Val MAE: 1.9804 | Val RMSE: 4.8414
Epoch 13/15 | Train Loss: 2.2903 | Val MAE: 2.0277 | Val RMSE: 5.0640
Epoch 14/15 | Train Loss: 2.1977 | Val MAE: 1.8369 | Val RMSE: 4.4689
Epoch 15/15 | Train Loss: 

best_val_mae,▁
epoch,▁▁▂▃▃▃▄▅▅▅▆▇▇▇█
test_loss,▁
test_mae,▁
test_rmse,▁
train_loss,█▄▃▂▂▂▂▂▂▁▁▁▁▁▁
val_loss,█▄▃▂▂▂▂▁▁▁▂▁▁▁▁
val_mae,█▄▃▂▂▂▂▁▁▁▂▁▁▁▁
val_rmse,█▄▃▃▂▂▂▂▁▁▂▁▁▁▁
best_val_mae,1.83692
epoch,15


...

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertModel LOAD REPORT from: distilbert-base-uncased
Key                     | Status     |  | 
------------------------+------------+--+-
vocab_layer_norm.bias   | UNEXPECTED |  | 
vocab_transform.bias    | UNEXPECTED |  | 
vocab_projector.bias    | UNEXPECTED |  | 
vocab_transform.weight  | UNEXPECTED |  | 
vocab_layer_norm.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Epoch 1/15 | Train Loss: 10.9087 | Val MAE: 7.4558 | Val RMSE: 16.9942
Epoch 2/15 | Train Loss: 6.2683 | Val MAE: 4.0333 | Val RMSE: 9.8676
Epoch 3/15 | Train Loss: 4.5734 | Val MAE: 3.3844 | Val RMSE: 8.3026
Epoch 4/15 | Train Loss: 3.9321 | Val MAE: 3.0881 | Val RMSE: 7.9645
Epoch 5/15 | Train Loss: 3.5782 | Val MAE: 2.5631 | Val RMSE: 6.9963
Epoch 6/15 | Train Loss: 3.3608 | Val MAE: 2.4627 | Val RMSE: 6.5031
Epoch 7/15 | Train Loss: 3.1638 | Val MAE: 2.4381 | Val RMSE: 6.1627
Epoch 8/15 | Train Loss: 2.9412 | Val MAE: 2.1772 | Val RMSE: 5.6756
Epoch 9/15 | Train Loss: 2.7726 | Val MAE: 2.1240 | Val RMSE: 5.5897
Epoch 10/15 | Train Loss: 2.6279 | Val MAE: 2.0190 | Val RMSE: 4.9535
Epoch 11/15 | Train Loss: 2.4976 | Val MAE: 2.4201 | Val RMSE: 5.6804
Epoch 12/15 | Train Loss: 2.3829 | Val MAE: 1.9529 | Val RMSE: 4.7061
Epoch 13/15 | Train Loss: 2.3024 | Val MAE: 1.9596 | Val RMSE: 4.7440
Epoch 14/15 | Train Loss: 2.1826 | Val MAE: 1.7727 | Val RMSE: 4.3669
Epoch 15/15 | Train Loss: 2

best_val_mae,▁
epoch,▁▁▂▃▃▃▄▅▅▅▆▇▇▇█
test_loss,▁
test_mae,▁
test_rmse,▁
train_loss,█▄▃▂▂▂▂▂▂▁▁▁▁▁▁
val_loss,█▄▃▃▂▂▂▁▁▁▂▁▁▁▁
val_mae,█▄▃▃▂▂▂▁▁▁▂▁▁▁▁
val_rmse,█▄▃▃▂▂▂▂▂▁▂▁▁▁▁
best_val_mae,1.77271
epoch,15


,experiment,text_column,text_model,text_scale,val_mae,test_mae,test_rmse,class_mae,mae_Tree,mae_Shrub,mae_Grass,mae_Crop,mae_Built-up,mae_Barren,mae_Water
0,swinv2_text_qwen3_4b_clean_distilbert-base-unc...,text_qwen3_4b_clean,distilbert-base-uncased,0.3,1.836919,1.936943,4.691902,"[2.4502, 0.948172, 4.462992, 2.7545323, 0.4679...",2.450200,0.948172,4.462992,2.754532,0.467947,2.117551,0.357207
1,swinv2_text_qwen3_4b_clean_distilbert-base-unc...,text_qwen3_4b_clean,distilbert-base-uncased,0.5,1.817483,1.899113,4.496265,"[2.3752997, 0.95082814, 4.4085374, 2.8849573, ...",2.375300,0.950828,4.408537,2.884957,0.515322,1.847036,0.311808
2,swinv2_text_qwen3_4b_clean_distilbert-base-unc...,text_qwen3_4b_clean,distilbert-base-uncased,0.7,1.757897,1.878618,4.518867,"[2.569227, 0.9502565, 4.1459947, 2.6390662, 0....",2.569227,0.950257,4.145995,2.639066,0.429315,2.022730,0.393734
3,swinv2_text_qwen3_4b_clean_distilbert-base-unc...,text_qwen3_4b_clean,distilbert-base-uncased,1.0,1.845992,1.944706,4.619997,"[2.6006758, 0.94978386, 4.638328, 2.8562286, 0...",2.600676,0.949784,4.638328,2.856229,0.397316,1.805128,0.365481
4,swinv2_text_qwen3_4b_clean_distilbert-base-unc...,text_qwen3_4b_clean,distilbert-base-uncased,1.5,1.772709,1.970701,4.455700,"[2.8254268, 0.9653588, 4.80339, 2.6154912, 0.4...",2.825427,0.965359,4.803390,2.615491,0.417539,1.840170,0.327536


## Text Scale Ablation — Overall MAE Summary

In [ ]:
image_only_row = next(r for r in results if r.get("text_model") in [None, "none"])
baseline_mae   = image_only_row["test_mae"]

scale_df["delta_vs_baseline"] = (baseline_mae - scale_df["test_mae"]).round(4)
scale_df["pct_improvement"]   = ((scale_df["delta_vs_baseline"] / baseline_mae) * 100).round(2)

print(f"Image-only baseline test MAE : {baseline_mae:.4f}\n")
print(f"Best config : {BEST_TEXT_COL}  +  {BEST_TEXT_MODEL}\n")

display_cols = ["text_scale", "val_mae", "test_mae", "test_rmse", "delta_vs_baseline", "pct_improvement"]
scale_df[display_cols]

Image-only baseline test MAE : 2.4836

Best config : text_qwen3_4b_clean  +  distilbert-base-uncased



,text_scale,val_mae,test_mae,test_rmse,delta_vs_baseline,pct_improvement
0,0.3,1.836919,1.936943,4.691902,0.5466,22.01
1,0.5,1.817483,1.899113,4.496265,0.5845,23.53
2,0.7,1.757897,1.878618,4.518867,0.6050,24.36
3,1.0,1.845992,1.944706,4.619997,0.5389,21.70
4,1.5,1.772709,1.970701,4.455700,0.5129,20.65


## Text Scale Ablation — Class-wise MAE

In [ ]:
class_mae_cols = [f"mae_{cls}" for cls in TARGET_COLS]
class_scale_df = scale_df[["text_scale"] + class_mae_cols].copy()
class_scale_df.columns = ["text_scale"] + TARGET_COLS

baseline_class_mae = np.array([image_only_row[f"mae_{cls}"] for cls in TARGET_COLS])

delta_rows = []
for _, row in class_scale_df.iterrows():
    row_mae = np.array([row[cls] for cls in TARGET_COLS])
    delta_rows.append(
        {"text_scale": row["text_scale"],
         **{cls: round(baseline_class_mae[i] - row_mae[i], 4) for i, cls in enumerate(TARGET_COLS)}}
    )

delta_class_df = pd.DataFrame(delta_rows)

print("Class-wise MAE per text scale")
print(class_scale_df.to_string(index=False))

print("\nClass-wise improvement vs image-only baseline (positive = better)")
print(delta_class_df.to_string(index=False))

Class-wise MAE per text scale
 text_scale     Tree    Shrub    Grass     Crop  Built-up   Barren    Water
        0.3 2.450200 0.948172 4.462992 2.754532  0.467947 2.117551 0.357207
        0.5 2.375300 0.950828 4.408537 2.884957  0.515322 1.847036 0.311808
        0.7 2.569227 0.950257 4.145995 2.639066  0.429315 2.022730 0.393734
        1.0 2.600676 0.949784 4.638328 2.856229  0.397316 1.805128 0.365481
        1.5 2.825427 0.965359 4.803390 2.615491  0.417539 1.840170 0.327536

Class-wise improvement vs image-only baseline (positive = better)
 text_scale    Tree  Shrub  Grass   Crop  Built-up  Barren   Water
        0.3  0.2700 0.0205 1.6234 1.9338    0.0512 -0.0721 -0.0003
        0.5  0.3449 0.0178 1.6778 1.8034    0.0038  0.1984  0.0451
        0.7  0.1509 0.0184 1.9404 2.0493    0.0898  0.0227 -0.0368
        1.0  0.1195 0.0189 1.4480 1.8321    0.1218  0.2403 -0.0085
        1.5 -0.1053 0.0033 1.2830 2.0729    0.1016  0.2052  0.0294


## Full Results Summary — All SwinV2 Experiments

In [ ]:
scale_result_experiments = {r["experiment"] for r in scale_results}

all_results_df = pd.DataFrame(results + scale_results).copy()

all_results_df["experiment_type"] = all_results_df.apply(
    lambda r: "image_only"  if pd.isna(r.get("text_model")) or r.get("text_model") in [None, "none"]
    else ("scale_sweep"     if r.get("experiment") in scale_result_experiments
    else  "multimodal_0.7"),
    axis=1
)

display_cols = [
    "experiment_type", "text_column", "text_model",
    "text_scale", "val_mae", "test_mae", "test_rmse"
]

all_results_df[display_cols].sort_values(
    ["experiment_type", "test_mae"]
).reset_index(drop=True)

,experiment_type,text_column,text_model,text_scale,val_mae,test_mae,test_rmse
0,image_only,NaN,NaN,NaN,NaN,2.483577,6.487426
1,multimodal_0.7,text_qwen3_4b_clean,sentence-transformers/all-MiniLM-L6-v2,0.7,1.813002,2.044218,4.662678
2,multimodal_0.7,hybrid_gemma3_4b_clean,sentence-transformers/all-MiniLM-L6-v2,0.7,2.000597,2.078193,5.153962
3,multimodal_0.7,text_qwen3_4b_clean,RemoteCLIP ViT-B-32,0.7,1.787881,2.095319,4.763024
4,multimodal_0.7,hybrid_gemma3_4b_clean,RemoteCLIP ViT-B-32,0.7,1.843484,2.112155,4.901970
5,multimodal_0.7,vision_gemma3_4b_clean,distilbert-base-uncased,0.7,2.143463,2.327044,6.156766
6,multimodal_0.7,vision_gemma3_4b_clean,sentence-transformers/all-MiniLM-L6-v2,0.7,2.187932,2.360168,6.202637
7,multimodal_0.7,hybrid_gemma3_4b_clean,distilbert-base-uncased,0.7,2.208799,2.363099,6.085603
8,multimodal_0.7,vision_gemma3_4b_clean,RemoteCLIP ViT-B-32,0.7,2.167152,2.408850,6.235519
9,scale_sweep,text_qwen3_4b_clean,distilbert-base-uncased,0.7,1.757897,1.878618,4.518867


In [ ]:
# Best configuration overall — across all experiments including scale sweep
all_multimodal = all_results_df[all_results_df["experiment_type"] != "image_only"]
best_overall = all_multimodal.loc[all_multimodal["val_mae"].idxmin()]

print("Best overall configuration (by validation MAE)")
print(f"  Experiment type : {best_overall['experiment_type']}")
print(f"  Text column     : {best_overall['text_column']}")
print(f"  Text model      : {best_overall['text_model']}")
print(f"  Text scale      : {best_overall['text_scale']}")
print(f"  Val MAE         : {best_overall['val_mae']:.4f}")
print(f"  Test MAE        : {best_overall['test_mae']:.4f}")
print(f"  Test RMSE       : {best_overall['test_rmse']:.4f}")

Best overall configuration (by validation MAE)
  Experiment type : scale_sweep
  Text column     : text_qwen3_4b_clean
  Text model      : distilbert-base-uncased
  Text scale      : 0.7
  Val MAE         : 1.7579
  Test MAE        : 1.8786
  Test RMSE       : 4.5189


In [ ]:
BACKBONE_NAME = CONFIG["model_name"].replace("_patch16_224", "").replace("_", "")

save_path = os.path.join(CONFIG["predictions_dir"], f"all_results_{BACKBONE_NAME}.csv")
all_results_df.to_csv(save_path, index=False)
print("Saved results to Drive")

Saved results to Drive
